# NovaMart Marketing Analytics Capstone
Run cells **top to bottom**. Upload `Project_2.xlsx` when prompted.

In [ ]:
# CELL 0.1 — Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import chi2_contingency, f_oneway, shapiro
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, classification_report,
                              confusion_matrix, ConfusionMatrixDisplay)
import warnings
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_colwidth', 40)
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.spines.top']   = False
plt.rcParams['axes.spines.right'] = False
print("All libraries loaded.")

## Upload File
Run this cell → click **Choose Files** → upload `Project_2.xlsx` → then continue.

In [ ]:
# CELL 0.2 — Upload
from google.colab import files
uploaded = files.upload()
excel_file = [f for f in uploaded.keys() if f.endswith('.xlsx')][0]
print(f"Uploaded: {excel_file}")

In [ ]:
# CELL 0.3 — Load raw sheets (all as strings to preserve messy values)
xl = pd.read_excel(excel_file, sheet_name=None, dtype=str)
customers_raw    = xl['customers'].copy()
campaigns_raw    = xl['campaigns'].copy()
leads_raw        = xl['leads'].copy()
sessions_raw     = xl['website_sessions'].copy()
transactions_raw = xl['transactions'].copy()
print("Sheets loaded:", {k: len(v) for k, v in xl.items()
                         if k not in ['README','data_dictionary']})

---
# Block 1 — Data Audit & Cleaning

In [ ]:
# CELL 1.1 — Helper functions

def clean_numeric(series):
    return (series.astype(str)
                  .str.replace(r'[^\d.\-]', '', regex=True)
                  .replace('', np.nan)
                  .pipe(pd.to_numeric, errors='coerce'))

def parse_bool(val):
    v = str(val).strip().lower()
    if v in ['yes', '1', 'y', 'true']:  return 1
    if v in ['no',  '0', 'n', 'false']: return 0
    return np.nan

def cramers_v(chi2, n, r, c):
    return np.sqrt(chi2 / (n * (min(r, c) - 1)))

def gini(arr):
    a = np.sort(np.array(arr, dtype=float))
    n = len(a); idx = np.arange(1, n + 1)
    return (2 * np.sum(idx * a) / (n * np.sum(a))) - (n + 1) / n

print("Helper functions defined.")

## 1A — Data Quality Scorecards

In [ ]:
# CELL 1.2 — Quality scorecard for every table

def scorecard(df, name):
    rows = []
    for col in df.columns:
        series      = df[col]
        pct_missing = series.isna().mean() * 100
        num         = pd.to_numeric(series, errors='coerce')
        pct_invalid = max(num.isna().mean() * 100 - pct_missing, 0)
        sample_bad  = series[
            series.astype(str).str.contains(r'[^\d\.\-]', regex=True, na=False)
            & ~series.isna()].unique()[:3].tolist()
        rows.append({'Table': name, 'Column': col,
                     'Dtype': str(series.dtype),
                     '% Missing': round(pct_missing, 1),
                     '% Invalid': round(pct_invalid, 1),
                     'Unique': series.nunique(),
                     'Sample Issues': str(sample_bad)})
    return pd.DataFrame(rows)

sc_all = pd.concat([scorecard(customers_raw,    'customers'),
                    scorecard(campaigns_raw,    'campaigns'),
                    scorecard(leads_raw,        'leads'),
                    scorecard(sessions_raw,     'website_sessions'),
                    scorecard(transactions_raw, 'transactions')], ignore_index=True)
print(sc_all.to_string(index=False))

## 1B — Clean Each Table

In [ ]:
# CELL 1.3 — Clean: CUSTOMERS
# Fixes: duplicates, age (out-of-range imputed by region median),
#        gender, region, city, loyalty_tier, device, signup_date

customers = customers_raw.copy()

before = len(customers)
customers = customers.drop_duplicates(subset='customer_id', keep='first')
print(f"Duplicates dropped: {before - len(customers)}")

customers['age'] = pd.to_numeric(customers['age'], errors='coerce')
invalid_age = (customers['age'] < 18) | (customers['age'] > 85)
print(f"Age out of range (< 18 or > 85): {invalid_age.sum()}")

customers['region_clean'] = customers['region'].str.strip().str.title()
region_medians = (customers.loc[~invalid_age, ['region_clean','age']]
                  .groupby('region_clean')['age'].median())
def impute_age(row):
    if pd.isna(row['age']) or row['age'] < 18 or row['age'] > 85:
        return region_medians.get(row['region_clean'], customers['age'].median())
    return row['age']
customers['age'] = customers.apply(impute_age, axis=1)

gmap = {'male':'Male','m':'Male','female':'Female','f':'Female',
        'non-binary':'Non-binary','prefer not say':'Prefer not to say'}
customers['gender'] = customers['gender'].str.strip().str.lower().map(gmap).fillna('Unknown')

rmap = {'south west':'South West','south-west':'South West','sw':'South West',
        'south south':'South South','s/south':'South South','south-south':'South South',
        'north central':'North Central','north cntrl':'North Central','n central':'North Central',
        'north west':'North West','nw':'North West','south east':'South East'}
customers['region'] = (customers['region'].str.strip().str.lower()
                       .map(rmap).fillna(customers['region'].str.strip().str.title()))

# City standardisation (Issue 6 fix)
# City standardisation — fix known misspellings
city_map = {' lagos':'Lagos','abjua':'Abuja','p/harcourt':'Port Harcourt'}
city_lower = customers['city'].str.strip().str.lower()
customers['city'] = city_lower.map(city_map).fillna(customers['city'].str.strip().str.title())

customers['loyalty_tier']    = customers['loyalty_tier'].str.strip().str.title()
customers['income_band']     = customers['income_band'].str.strip()
dmap = {'phone':'Mobile','mobile':'Mobile','tab':'Tablet','desktop':'Desktop','tablet':'Tablet'}
customers['preferred_device'] = (customers['preferred_device'].str.strip().str.lower()
                                  .map(dmap).fillna(customers['preferred_device'].str.strip().str.title()))
customers['signup_date'] = pd.to_datetime(customers['signup_date'], errors='coerce')

print(f"Customers clean: {len(customers)} rows")

In [ ]:
# CELL 1.4 — Clean: CAMPAIGNS

campaigns = campaigns_raw.copy()
for c in ['budget_usd','spend_usd','impressions','clicks']:
    campaigns[c] = clean_numeric(campaigns[c])
neg_spend = (campaigns['spend_usd'] < 0).sum()
campaigns.loc[campaigns['spend_usd'] < 0, 'spend_usd'] = np.nan
print(f"Negative spend set to NaN: {neg_spend}")
campaigns['ctr_flag'] = campaigns['clicks'] > campaigns['impressions']
print(f"Impossible CTR (clicks > impressions): {campaigns['ctr_flag'].sum()}")

chmap = {'paid social':'Paid Social','paid-social':'Paid Social',
         'paid search':'Paid Search','search':'Paid Search',
         'email':'Email','e-mail':'Email','affiliate':'Affiliate',
         'influencer':'Influencer','influencers':'Influencer','display':'Display'}
campaigns['channel'] = (campaigns['channel'].str.strip().str.lower()
                        .map(chmap).fillna(campaigns['channel'].str.strip().str.title()))
tr_map = {'all':'All','nationwide':'All','sw':'South West','south west':'South West',
          'south south':'South South','s/s':'South South',
          'north central':'North Central','north c.':'North Central','north west':'North West'}
campaigns['target_region'] = (campaigns['target_region'].str.strip().str.lower()
                               .map(tr_map).fillna(campaigns['target_region'].str.strip().str.title()))
campaigns['creative_type'] = campaigns['creative_type'].str.strip().str.title()
campaigns['objective']     = campaigns['objective'].str.strip().str.title()
campaigns['start_date']    = pd.to_datetime(campaigns['start_date'], errors='coerce')
campaigns['end_date']      = pd.to_datetime(campaigns['end_date'],   errors='coerce')
print(f"Campaigns clean: {len(campaigns)} rows")

In [ ]:
# CELL 1.5 — Clean: LEADS

leads = leads_raw.copy()
leads['converted_30d'] = leads['converted_30d'].apply(parse_bool)

disc_words = {'fifteen':'15','ten':'10','five':'5','twenty':'20','twenty-five':'25'}
leads['discount_offered_pct'] = (leads['discount_offered_pct'].astype(str)
                                  .replace(disc_words, regex=False))
leads['discount_offered_pct'] = clean_numeric(leads['discount_offered_pct']).clip(lower=0, upper=100)
leads['acquisition_cost_usd'] = clean_numeric(leads['acquisition_cost_usd'])
leads['lead_score'] = leads['lead_score'].replace('?', np.nan)
leads['lead_score'] = pd.to_numeric(leads['lead_score'], errors='coerce')

src_map = {'google':'Google','meta ads':'Meta','meta':'Meta','crm':'CRM',
           'partner':'Partner','creator':'Creator','programmatic':'Programmatic','email':'Email'}
leads['lead_source'] = (leads['lead_source'].str.strip().str.lower()
                        .map(src_map).fillna(leads['lead_source'].str.strip().str.title()))
leads['observed_region'] = (leads['observed_region'].str.strip().str.lower()
                             .map(rmap).fillna(leads['observed_region'].str.strip().str.title()))
leads['landing_page']    = leads['landing_page'].str.strip()
leads['lead_date']       = pd.to_datetime(leads['lead_date'],       errors='coerce')
leads['conversion_date'] = pd.to_datetime(leads['conversion_date'], errors='coerce')
print(f"Leads clean: {len(leads)} rows")
print(f"converted_30d:\n{leads['converted_30d'].value_counts()}")

In [ ]:
# CELL 1.6 — Clean: WEBSITE SESSIONS

sessions = sessions_raw.copy()
for col in ['bounce','add_to_cart','checkout_started']:
    sessions[col] = sessions[col].apply(parse_bool)
sessions['pages_viewed']     = pd.to_numeric(sessions['pages_viewed'],     errors='coerce')
sessions['time_on_site_sec'] = pd.to_numeric(sessions['time_on_site_sec'], errors='coerce')
sessions.loc[sessions['time_on_site_sec'] < 0,    'time_on_site_sec'] = np.nan
sessions.loc[sessions['time_on_site_sec'] > 7200, 'time_on_site_sec'] = np.nan

sessions['device'] = (sessions['device'].str.strip().str.lower()
                      .map(dmap).fillna(sessions['device'].str.strip().str.title()))
cg_map = {'paid social':'Paid Social','paid search':'Paid Search','email':'Email',
          'influencer':'Influencer','affiliate':'Affiliate','referral':'Referral',
          'direct':'Direct','organic':'Organic','display':'Display','display ads':'Display'}
sessions['channel_group'] = (sessions['channel_group'].str.strip().str.lower()
                              .map(cg_map).fillna(sessions['channel_group'].str.strip().str.title()))
sessions['geo_region']   = (sessions['geo_region'].str.strip().str.lower()
                             .map(rmap).fillna(sessions['geo_region'].str.strip().str.title()))
sessions['session_date'] = pd.to_datetime(sessions['session_date'], errors='coerce')
print(f"Sessions clean: {len(sessions)} rows")

In [ ]:
# CELL 1.7 — Clean: TRANSACTIONS

transactions = transactions_raw.copy()
transactions['revenue_usd'] = clean_numeric(transactions['revenue_usd'])
neg_rev = (transactions['revenue_usd'] < 0).sum()
p99_rev = transactions['revenue_usd'].quantile(0.99)
transactions.loc[transactions['revenue_usd'] < 0, 'revenue_usd'] = np.nan
transactions['revenue_usd'] = transactions['revenue_usd'].clip(upper=p99_rev)
print(f"Negative revenue → NaN: {neg_rev} | 99th pctile cap: ${p99_rev:.2f}")

disc_words2 = {'twenty':'20','ten':'10','five':'5','fifteen':'15','twenty-five':'25'}
transactions['discount_pct'] = (transactions['discount_pct'].astype(str)
                                 .replace(disc_words2, regex=False))
transactions['discount_pct'] = clean_numeric(transactions['discount_pct']).clip(lower=0, upper=100).fillna(0)
transactions['returned_flag'] = transactions['returned_flag'].apply(parse_bool)
transactions['units']         = pd.to_numeric(transactions['units'], errors='coerce')

cat_map = {'snacks':'Snacks','home care':'Home Care','baby':'Baby','baby care':'Baby',
           'supplements':'Supplements','personal care':'Personal Care',
           'personal-care':'Personal Care','beverages':'Beverages','beverage':'Beverages'}
transactions['product_category'] = (transactions['product_category'].str.strip().str.lower()
                                     .map(cat_map).fillna(transactions['product_category'].str.strip().str.title()))
pay_map = {'cash on delivery':'Cash on Delivery','cod':'Cash on Delivery','cash':'Cash on Delivery',
           'transfer':'Bank Transfer','bank transfer':'Bank Transfer','card':'Card','wallet':'Wallet'}
transactions['payment_type'] = (transactions['payment_type'].str.strip().str.lower()
                                 .map(pay_map).fillna(transactions['payment_type'].str.strip().str.title()))
lt_map = {'influencer':'Influencer','paid search':'Paid Search','search':'Paid Search',
          'direct':'Direct','paid social':'Paid Social','email':'Email',
          'affiliate':'Affiliate','organic':'Organic'}
transactions['marketing_channel_last_touch'] = (
    transactions['marketing_channel_last_touch'].str.strip().str.lower()
    .map(lt_map).fillna(transactions['marketing_channel_last_touch'].str.strip().str.title()))
transactions['order_date'] = pd.to_datetime(transactions['order_date'], errors='coerce')
print(f"Transactions clean: {len(transactions)} rows")

## 1C — Join Audit

In [ ]:
# CELL 1.8 — Join audit

def join_audit(left_df, right_df, key, ln, rn):
    lk = set(left_df[key].dropna().unique())
    rk = set(right_df[key].dropna().unique())
    matched = lk & rk
    print(f"\n{ln} ↔ {rn}  (key: '{key}')")
    print(f"  Left keys     : {len(lk)}")
    print(f"  Right keys    : {len(rk)}")
    print(f"  Matched       : {len(matched)}  ({len(matched)/len(lk)*100:.1f}%)")
    print(f"  Orphaned left : {len(lk - rk)}")

join_audit(customers,  leads,        'customer_id', 'customers', 'leads')
join_audit(customers,  transactions, 'customer_id', 'customers', 'transactions')
join_audit(campaigns,  leads,        'campaign_id', 'campaigns', 'leads')
join_audit(campaigns,  sessions,     'campaign_id', 'campaigns', 'sessions')

## 1D — Build Analytical Base Table (ABT)

In [ ]:
# CELL 1.9 — Customer-level summaries

txn_sum = (transactions.dropna(subset=['customer_id'])
           .groupby('customer_id')
           .agg(total_revenue    = ('revenue_usd',   'sum'),
                total_orders     = ('order_id',       'count'),
                avg_order_value  = ('revenue_usd',    'mean'),
                total_units      = ('units',          'sum'),
                return_count     = ('returned_flag',  'sum'),
                avg_discount_t   = ('discount_pct',   'mean'))
           .reset_index())
txn_sum['customer_ltv']    = txn_sum['total_revenue']
txn_sum['is_repeat_buyer'] = (txn_sum['total_orders'] >= 2).astype(int)

sess_sum = (sessions.dropna(subset=['customer_id'])
            [['session_id','customer_id','pages_viewed','time_on_site_sec',
              'add_to_cart','checkout_started','bounce']]
            .groupby('customer_id')
            .agg(total_sessions = ('session_id',       'count'),
                 avg_pages      = ('pages_viewed',     'mean'),
                 avg_time       = ('time_on_site_sec', 'mean'),
                 any_cart       = ('add_to_cart',      'max'),
                 any_checkout   = ('checkout_started', 'max'),
                 bounce_rate    = ('bounce',           'mean'))
            .reset_index())
sess_sum['sqs_raw'] = (sess_sum['avg_pages'] * 0.3 +
                       sess_sum['avg_time'] / 60 * 0.3 +
                       sess_sum['any_cart']    * 0.2 +
                       sess_sum['any_checkout']* 0.2)
mn, mx = sess_sum['sqs_raw'].min(), sess_sum['sqs_raw'].max()
sess_sum['session_quality_score'] = (sess_sum['sqs_raw'] - mn) / (mx - mn + 1e-9)

print("Transaction summary:", txn_sum.shape)
print("Session summary    :", sess_sum.shape)

In [ ]:
# CELL 1.10 — Build ABT

abt = (leads
       .merge(customers[['customer_id','gender','age','city','region','signup_date',
                          'preferred_device','loyalty_tier','email_opt_in','income_band']],
              on='customer_id', how='left')
       .merge(campaigns[['campaign_id','channel','objective','spend_usd','budget_usd',
                          'impressions','clicks','creative_type','target_region']],
              on='campaign_id', how='left')
       .merge(txn_sum,  on='customer_id', how='left')
       .merge(sess_sum, on='customer_id', how='left'))

abt['revenue_per_unit']      = abt['total_revenue'] / abt['total_units'].replace(0, np.nan)
abt['days_to_convert']       = (abt['conversion_date'] - abt['lead_date']).dt.days
abt['discount_flag']         = (abt['discount_offered_pct'] > 0).astype(int)
abt['is_repeat_buyer']       = abt['is_repeat_buyer'].fillna(0).astype(int)
abt['customer_ltv']          = abt['customer_ltv'].fillna(0)
abt['session_quality_score'] = abt['session_quality_score'].fillna(0)

abt_dev = abt.merge(
    sessions.dropna(subset=['customer_id'])
    .groupby('customer_id')['device'].first().reset_index(),
    on='customer_id', how='left')
abt_txn = abt.merge(
    transactions[['customer_id','product_category','returned_flag','revenue_usd']]
    .dropna(subset=['customer_id']),
    on='customer_id', how='left')
cust_ltv = customers.merge(
    txn_sum[['customer_id','customer_ltv']], on='customer_id', how='left')
cust_ltv['customer_ltv'] = cust_ltv['customer_ltv'].fillna(0)

print(f"ABT shape: {abt.shape}")

---
# Block 2 — Business KPI Dashboard

In [ ]:
# CELL 2.1 — Overall KPIs

total_leads      = len(abt)
conv_rate        = abt['converted_30d'].mean() * 100
avg_cpa          = abt[abt['converted_30d']==1]['acquisition_cost_usd'].mean()
total_rev        = transactions['revenue_usd'].sum()
total_spend      = campaigns['spend_usd'].sum()
roas             = total_rev / total_spend
aov              = transactions['revenue_usd'].mean()
repeat_rate      = txn_sum['is_repeat_buyer'].mean() * 100
return_rate      = transactions['returned_flag'].mean() * 100
avg_days_convert = abt['days_to_convert'].dropna().mean()
lead_rev_ratio   = abt[abt['converted_30d']==1]['customer_ltv'].sum() / total_leads

kpis = pd.DataFrame({
    'KPI': ['Total Leads','Conversion Rate (%)','Avg CPA (USD)',
            'Total Revenue (USD)','Total Spend (USD)','ROAS',
            'Avg Order Value (USD)','Repeat Buyer Rate (%)','Return Rate (%)',
            'Avg Days to Convert','Lead-to-Revenue Ratio (USD)'],
    'Value': [f"{total_leads:,}", f"{conv_rate:.1f}%", f"${avg_cpa:.2f}",
              f"${total_rev:,.2f}", f"${total_spend:,.2f}", f"{roas:.2f}",
              f"${aov:.2f}", f"{repeat_rate:.1f}%", f"{return_rate:.1f}%",
              f"{avg_days_convert:.1f} days", f"${lead_rev_ratio:.2f}"]})
print("=" * 50)
print("    NOVAMART — EXECUTIVE KPI DASHBOARD")
print("=" * 50)
print(kpis.to_string(index=False))

In [ ]:
# CELL 2.2 — KPI breakdown by key dimensions

def kpi_pivot(group_col):
    result = (abt.groupby(group_col)
              .agg(total_leads     = ('lead_id',              'count'),
                   conversion_rate = ('converted_30d',        'mean'),
                   avg_cpa         = ('acquisition_cost_usd', 'mean'),
                   avg_ltv         = ('customer_ltv',         'mean'),
                   repeat_rate     = ('is_repeat_buyer',      'mean'),
                   avg_days        = ('days_to_convert',      'mean'),
                   avg_discount    = ('discount_offered_pct', 'mean'))
              .round(3)
              .sort_values('conversion_rate', ascending=False))
    result['conversion_rate'] = (result['conversion_rate'] * 100).round(1)
    result['repeat_rate']     = (result['repeat_rate']     * 100).round(1)
    return result

for dim in ['channel','observed_region','loyalty_tier','objective','income_band']:
    if abt[dim].dropna().nunique() > 1:
        print(f"\n{'='*65}\n  KPI BREAKDOWN BY: {dim.upper()}\n{'='*65}")
        print(kpi_pivot(dim).to_string())

## 2A — City-Level Analysis (Issue 6)

In [ ]:
# CELL 2.3 — City-level KPI analysis
# Merges customer city into abt for geographic breakdowns

# city is already in abt from the Block 1 merge
abt_city = abt.copy()

city_kpis = (abt_city.groupby('city')
             .agg(total_leads     = ('lead_id',              'count'),
                  conversion_rate = ('converted_30d',        'mean'),
                  avg_ltv         = ('customer_ltv',         'mean'),
                  avg_cpa         = ('acquisition_cost_usd', 'mean'),
                  repeat_rate     = ('is_repeat_buyer',      'mean'))
             .round(3)
             .sort_values('conversion_rate', ascending=False))
city_kpis['conversion_rate'] = (city_kpis['conversion_rate'] * 100).round(1)
city_kpis['repeat_rate']     = (city_kpis['repeat_rate']     * 100).round(1)
print("=== KPI BREAKDOWN BY CITY ===")
print(city_kpis.to_string())

# Bar chart: top 10 cities by total leads
top10 = city_kpis.nlargest(10, 'total_leads')
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].bar(top10.index, top10['total_leads'], color='steelblue')
axes[0].set_title('Top 10 Cities by Lead Volume')
axes[0].tick_params(axis='x', rotation=45)
axes[0].set_ylabel('Total Leads')

top10_conv = city_kpis.nlargest(10, 'conversion_rate')
axes[1].bar(top10_conv.index, top10_conv['conversion_rate'], color='coral')
axes[1].set_title('Top 10 Cities by Conversion Rate')
axes[1].tick_params(axis='x', rotation=45)
axes[1].set_ylabel('Conversion Rate (%)')
plt.tight_layout(); plt.show()

---
# Block 3 — Chi-Square Tests

In [ ]:
# CELL 3.1 — Chi-square tests (10 tests)

def chi_test(df, col1, col2, label):
    sub = df[[col1, col2]].dropna()
    ct  = pd.crosstab(sub[col1], sub[col2])
    if ct.shape[0] < 2 or ct.shape[1] < 2:
        print(f"\n{label}: Skipped"); return
    chi2, p, dof, _ = chi2_contingency(ct)
    n  = sub.shape[0]
    cv = cramers_v(chi2, n, ct.shape[0], ct.shape[1])
    strength = 'Weak' if cv < 0.1 else ('Moderate' if cv < 0.3 else 'Strong')
    sig = 'SIGNIFICANT ✓' if p < 0.05 else 'Not significant'
    print(f"\n{'─'*62}")
    print(f"TEST: {label}")
    print(f"  H0: {col1} and {col2} are independent")
    print(f"  H1: There is an association between {col1} and {col2}")
    print(f"  Chi² = {chi2:.3f} | df = {dof} | p = {p:.4f}")
    print(f"  Cramér's V = {cv:.3f}  ({strength} effect)")
    print(f"  Conclusion (α=0.05): {sig}")

chi_test(abt,     'channel',        'converted_30d',    '1. Channel × Conversion')
chi_test(abt_dev, 'device',         'converted_30d',    '2. Device × Conversion')
chi_test(abt,     'creative_type',  'converted_30d',    '3. Creative Type × Conversion')
chi_test(abt,     'observed_region','is_repeat_buyer',  '4. Region × Repeat Buyer')
chi_test(abt_txn, 'loyalty_tier',   'returned_flag',    '5. Loyalty Tier × Returns')
chi_test(abt,     'discount_flag',  'converted_30d',    '6. Discount Flag × Conversion')
chi_test(abt,     'objective',      'is_repeat_buyer',  '7. Campaign Objective × Repeat Buyer')
chi_test(abt_txn, 'gender',         'product_category', '8. Gender × Product Category')
chi_test(abt,     'income_band',    'loyalty_tier',     '9. Income Band × Loyalty Tier')
chi_test(abt,     'landing_page',   'converted_30d',    '10. Landing Page × Conversion')

---
# Block 4 — One-Way ANOVA

In [ ]:
# CELL 4.1 — ANOVA helper with Tukey HSD

def one_way_anova(df, group_col, value_col, label):
    sub    = df[[group_col, value_col]].dropna()
    groups = [g[value_col].values for _, g in sub.groupby(group_col) if len(g) > 1]
    if len(groups) < 2:
        print(f"\n{label}: Skipped"); return
    f, p = f_oneway(*groups)
    sig  = 'SIGNIFICANT ✓' if p < 0.05 else 'Not significant'
    print(f"\n{'─'*62}")
    print(f"ONE-WAY ANOVA: {label}")
    print(f"  H0: All group means are equal")
    for name, g in sub.groupby(group_col):
        print(f"    {str(name):25s} — Mean: {g[value_col].mean():.2f}  N: {len(g)}")
    print(f"  F = {f:.4f} | p = {p:.4f} → {sig}")
    if p < 0.05:
        mc        = pairwise_tukeyhsd(sub[value_col], sub[group_col], alpha=0.05)
        sig_pairs = [(r[0], r[1]) for r in mc.summary().data[1:] if r[6]]
        if sig_pairs:
            print("  Tukey HSD — Significantly different pairs:")
            for pair in sig_pairs: print(f"    • {pair[0]}  vs  {pair[1]}")
        else:
            print("  Tukey HSD — No specific pair significantly different after correction")

In [ ]:
# CELL 4.2 — Run all 6 One-Way ANOVAs

one_way_anova(abt_txn,  'observed_region', 'revenue_usd',           '1. Revenue across Regions')
one_way_anova(cust_ltv, 'loyalty_tier',    'customer_ltv',          '2. LTV across Loyalty Tiers')
one_way_anova(abt,      'channel',         'days_to_convert',       '3. Days to Convert across Channels')
one_way_anova(abt_dev,  'device',          'session_quality_score', '4. Session Quality across Devices')
one_way_anova(abt,      'objective',       'total_revenue',         '5. Revenue across Campaign Objectives')
one_way_anova(abt,      'income_band',     'avg_order_value',       '6. AOV across Income Bands')

---
# Block 5 — Simple Linear Regression (SLR)

In [ ]:
# CELL 5.1 — SLR helper
# Uses sklearn LinearRegression for simplicity; reports R², coefficient,
# p-value (via scipy), 95% CI, and scatter plot with regression line.

from sklearn.linear_model import LinearRegression as LR_sklearn
from scipy.stats import t as t_dist

def slr(x_series, y_series, x_label, y_label, title):
    df2 = pd.DataFrame({'x': x_series, 'y': y_series}).dropna()
    if len(df2) < 10:
        print(f"Skipped — insufficient data"); return

    X = df2[['x']].values
    y = df2['y'].values
    n = len(y)

    model = LR_sklearn()
    model.fit(X, y)
    coef  = model.coef_[0]
    y_hat = model.predict(X)

    # R²
    ss_res = np.sum((y - y_hat) ** 2)
    ss_tot = np.sum((y - y.mean()) ** 2)
    r2     = 1 - ss_res / ss_tot if ss_tot > 0 else 0

    # Standard error of coefficient and p-value
    se_sq   = (ss_res / (n - 2)) / (np.sum((df2['x'] - df2['x'].mean()) ** 2) + 1e-9)
    se_coef = np.sqrt(se_sq)
    t_stat  = coef / se_coef
    p_val   = 2 * (1 - t_dist.cdf(abs(t_stat), df=n - 2))

    # 95% CI
    t_crit = t_dist.ppf(0.975, df=n - 2)
    ci_lo  = coef - t_crit * se_coef
    ci_hi  = coef + t_crit * se_coef

    print(f"\n{'─'*62}")
    print(f"SLR: {title}")
    print(f"  R²    = {r2:.4f}")
    print(f"  Coef  = {coef:.4f}  |  p = {p_val:.4f}")
    print(f"  95% CI: [{ci_lo:.4f}, {ci_hi:.4f}]")
    print(f"  → A 1-unit increase in {x_label} is associated with "
          f"a {coef:+.4f}-unit change in {y_label}")

    plt.figure(figsize=(7, 4))
    plt.scatter(df2['x'], df2['y'], alpha=0.3, s=15, color='steelblue')
    xfit = np.linspace(df2['x'].min(), df2['x'].max(), 200)
    plt.plot(xfit, model.intercept_ + coef * xfit, color='tomato', linewidth=2)
    plt.xlabel(x_label); plt.ylabel(y_label)
    plt.title(f"{title}  |  R²={r2:.3f}, p={p_val:.4f}")
    plt.tight_layout(); plt.show()

In [ ]:
# CELL 5.2 — Campaign-level SLRs

camp_leads_agg = leads.groupby('campaign_id').agg(total_leads=('lead_id','count')).reset_index()
camp_data = campaigns.merge(camp_leads_agg, on='campaign_id', how='left')

slr(camp_data['spend_usd'], camp_data['total_leads'],
    'Campaign Spend (USD)', 'Total Leads', '1. Campaign Spend → Leads Generated')
slr(campaigns['impressions'], campaigns['clicks'],
    'Impressions', 'Clicks', '7. Impressions → Clicks')

In [ ]:
# CELL 5.3 — Lead/session/transaction level SLRs

converted_abt = abt[abt['converted_30d'] == 1]
slr(converted_abt['lead_score'], converted_abt['total_revenue'],
    'Lead Score', 'Total Revenue (USD)', '2. Lead Score → Revenue (Converted Leads)')
slr(abt['avg_time'], abt['converted_30d'],
    'Avg Time on Site (sec)', 'Converted (0/1)', '3. Time on Site → Conversion')
slr(sessions['pages_viewed'], sessions['add_to_cart'],
    'Pages Viewed', 'Add to Cart (0/1)', '4. Pages Viewed → Add to Cart')
slr(transactions['discount_pct'], transactions['revenue_usd'],
    'Discount %', 'Revenue (USD)', '5. Discount % → Revenue')
slr(abt['acquisition_cost_usd'], abt['customer_ltv'],
    'Acquisition Cost (USD)', 'Customer LTV (USD)', '6. Acquisition Cost → LTV')
slr(abt['session_quality_score'], abt['is_repeat_buyer'],
    'Session Quality Score', 'Is Repeat Buyer (0/1)', '8. Session Quality → Repeat Buyer')

---
# Block 6 — Multiple Linear Regression (MLR)

**Note on encoding:** Nominal categorical variables (channel, region, product_category, etc.)
are one-hot encoded using `pd.get_dummies` with `drop_first=True` to avoid the dummy variable trap.
This prevents OLS from incorrectly treating category labels as ordered numbers.

In [ ]:
# CELL 6.1 — MLR diagnostics helper
# Uses sklearn LinearRegression; reports R², adjusted R², residual plot,
# Q-Q plot, Shapiro-Wilk normality test, and top 3 predictors by |coef|.

from sklearn.linear_model import LinearRegression as LR_sklearn
from sklearn.metrics import r2_score

def mlr_diagnostics(model, X_df, y_series, feature_names, title):
    y_hat     = model.predict(X_df)
    residuals = y_series.values - y_hat
    n, p      = X_df.shape
    r2        = r2_score(y_series, y_hat)
    adj_r2    = 1 - (1 - r2) * (n - 1) / (n - p - 1)

    print(f"\n{'═'*62}")
    print(f"MODEL: {title}")
    print(f"  R²          = {r2:.4f}")
    print(f"  Adjusted R² = {adj_r2:.4f}")
    print(f"  N           = {n}  |  Features = {p}")

    # Coefficients table
    coef_df = pd.DataFrame({'Feature': feature_names, 'Coefficient': model.coef_}).sort_values('Coefficient', ascending=False)
    print("\n  Coefficients:")
    print(coef_df.to_string(index=False))

    # Residual plot + Q-Q plot
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].scatter(y_hat, residuals, alpha=0.3, s=15, color='steelblue')
    axes[0].axhline(0, color='tomato', linewidth=1)
    axes[0].set_xlabel('Fitted Values'); axes[0].set_ylabel('Residuals')
    axes[0].set_title(f'{title} — Residual Plot')
    sm.qqplot(residuals, line='s', ax=axes[1], alpha=0.4)
    axes[1].set_title('Q-Q Plot of Residuals')
    plt.tight_layout(); plt.show()

    # Shapiro-Wilk (normality of residuals)
    sw_stat, sw_p = shapiro(residuals[:5000])
    print(f"  Shapiro-Wilk: stat={sw_stat:.4f}, p={sw_p:.4f} → "
          f"{'Residuals NOT normal' if sw_p < 0.05 else 'Approximately normal'}")

    # Top 3 predictors
    top3 = coef_df.iloc[:3]['Feature'].tolist()
    print(f"  Top 3 predictors (by coefficient magnitude): {top3}")

In [ ]:
# CELL 6.2 — Model A: Predict customer_ltv
# One-hot encode nominal categoricals (drop_first=True to avoid dummy trap)

from sklearn.linear_model import LinearRegression as LR_sklearn

modA_raw = abt[['customer_ltv','acquisition_cost_usd','days_to_convert',
                'session_quality_score','discount_flag','is_repeat_buyer',
                'loyalty_tier','observed_region','channel']].dropna().copy()

modA = pd.get_dummies(modA_raw,
                       columns=['loyalty_tier','observed_region','channel'],
                       drop_first=True).astype(float)

y_A      = modA['customer_ltv'].values
X_A_df   = modA.drop(columns='customer_ltv')
feat_A   = X_A_df.columns.tolist()
X_A      = X_A_df.values

mA = LR_sklearn()
mA.fit(X_A, y_A)
mlr_diagnostics(mA, X_A_df, modA['customer_ltv'], feat_A, 'Model A — Predict Customer LTV')

In [ ]:
# CELL 6.3 — Model B: Predict revenue_usd per transaction
# One-hot encode nominal categoricals (drop_first=True)

from sklearn.linear_model import LinearRegression as LR_sklearn

modB_raw = transactions[['revenue_usd','discount_pct','units',
                          'product_category','payment_type',
                          'marketing_channel_last_touch']].dropna().copy()

modB = pd.get_dummies(modB_raw,
                       columns=['product_category','payment_type',
                                'marketing_channel_last_touch'],
                       drop_first=True).astype(float)

y_B    = modB['revenue_usd'].values
X_B_df = modB.drop(columns='revenue_usd')
feat_B = X_B_df.columns.tolist()

mB = LR_sklearn()
mB.fit(X_B_df.values, y_B)
mlr_diagnostics(mB, X_B_df, modB['revenue_usd'], feat_B, 'Model B — Predict Revenue per Transaction')

In [ ]:
# CELL 6.4 — Model C: Predict days_to_convert
# One-hot encode nominal categoricals (drop_first=True)

from sklearn.linear_model import LinearRegression as LR_sklearn

modC_raw = abt[['days_to_convert','lead_score','discount_offered_pct',
                'channel','landing_page','objective',
                'session_quality_score']].dropna().copy()

modC = pd.get_dummies(modC_raw,
                       columns=['channel','landing_page','objective'],
                       drop_first=True).astype(float)

y_C    = modC['days_to_convert'].values
X_C_df = modC.drop(columns='days_to_convert')
feat_C = X_C_df.columns.tolist()

mC = LR_sklearn()
mC.fit(X_C_df.values, y_C)
mlr_diagnostics(mC, X_C_df, modC['days_to_convert'], feat_C, 'Model C — Predict Days to Convert')

---
# Block 7 — Gini Coefficient, Lorenz Curve & Pareto Analysis

In [ ]:
# CELL 7.1 — Gini + Lorenz curve for Customer LTV

ltv_vals  = txn_sum['customer_ltv'].dropna().values
ltv_vals  = ltv_vals[ltv_vals >= 0]
g_overall = gini(ltv_vals)
print(f"Gini Coefficient of Customer LTV: {g_overall:.4f}")

def lorenz_curve(arr):
    a = np.sort(arr)
    return (np.linspace(0, 1, len(a) + 1),
            np.concatenate([[0], np.cumsum(a) / np.sum(a)]))

x, y = lorenz_curve(ltv_vals)
plt.figure(figsize=(7, 6))
plt.plot(x, y, color='steelblue', linewidth=2, label=f'LTV Lorenz (Gini={g_overall:.3f})')
plt.plot([0,1],[0,1], 'k--', linewidth=1, label='Perfect Equality')
plt.fill_between(x, x, y, alpha=0.15, color='steelblue')
plt.xlabel('Cumulative Share of Customers')
plt.ylabel('Cumulative Share of LTV')
plt.title('Lorenz Curve — Customer LTV')
plt.legend(); plt.tight_layout(); plt.show()

In [ ]:
# CELL 7.2 — Gini by Region and Loyalty Tier

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
region_ginis = {}
for name, grp in cust_ltv.dropna(subset=['region','customer_ltv']).groupby('region'):
    vals = grp['customer_ltv'].values
    if len(vals) > 2 and vals.sum() > 0:
        region_ginis[name] = gini(vals)
        x2, y2 = lorenz_curve(vals)
        axes[0].plot(x2, y2, label=f"{name} (G={region_ginis[name]:.3f})")
axes[0].plot([0,1],[0,1],'k--')
axes[0].set_title('Lorenz by Region'); axes[0].legend(fontsize=8)

loyalty_ginis = {}
for name, grp in cust_ltv.dropna(subset=['loyalty_tier','customer_ltv']).groupby('loyalty_tier'):
    vals = grp['customer_ltv'].values
    if len(vals) > 2 and vals.sum() > 0:
        loyalty_ginis[name] = gini(vals)
        x2, y2 = lorenz_curve(vals)
        axes[1].plot(x2, y2, label=f"{name} (G={loyalty_ginis[name]:.3f})")
axes[1].plot([0,1],[0,1],'k--')
axes[1].set_title('Lorenz by Loyalty Tier'); axes[1].legend(fontsize=8)
plt.tight_layout(); plt.show()
print("Gini by Region  :", {k: round(v,3) for k,v in region_ginis.items()})
print("Gini by Loyalty :", {k: round(v,3) for k,v in loyalty_ginis.items()})

In [ ]:
# CELL 7.3 — Pareto analysis

ltv_sorted = np.sort(ltv_vals)[::-1]
cum_ltv    = np.cumsum(ltv_sorted) / ltv_sorted.sum()
n_for_80   = np.argmax(cum_ltv >= 0.80) + 1
pct_cust   = n_for_80 / len(ltv_sorted) * 100
print(f"Pareto: {pct_cust:.1f}% of customers account for 80% of total LTV")

txn_sum_d = txn_sum.copy()
txn_sum_d['ltv_decile'] = pd.qcut(txn_sum_d['customer_ltv'], q=10,
                                    labels=[f'D{i}' for i in range(1,11)])
decile_ltv = txn_sum_d.groupby('ltv_decile', observed=True)['customer_ltv'].sum()
plt.figure(figsize=(9, 5))
decile_ltv.plot(kind='bar', color='steelblue', edgecolor='black')
plt.title('Total LTV by Decile  (D1=Lowest, D10=Highest)')
plt.xlabel('LTV Decile'); plt.ylabel('Total LTV (USD)')
plt.xticks(rotation=0); plt.tight_layout(); plt.show()

In [ ]:
# CELL 7.4 — Bottom 20% LTV: net negative acquisition?

bottom_20_cut = txn_sum['customer_ltv'].quantile(0.20)
bottom_20     = txn_sum[txn_sum['customer_ltv'] <= bottom_20_cut].copy()
top_80        = txn_sum[txn_sum['customer_ltv'] >  bottom_20_cut].copy()
acq_merge     = leads[['customer_id','acquisition_cost_usd']].dropna().drop_duplicates('customer_id')
bot_acq = bottom_20.merge(acq_merge, on='customer_id', how='left')
top_acq = top_80.merge(acq_merge,    on='customer_id', how='left')
net_bot = bottom_20['customer_ltv'].mean() - bot_acq['acquisition_cost_usd'].mean()
net_top = top_80['customer_ltv'].mean()    - top_acq['acquisition_cost_usd'].mean()
print(f"Bottom 20% — Avg LTV: ${bottom_20['customer_ltv'].mean():.2f} | "
      f"Avg CPA: ${bot_acq['acquisition_cost_usd'].mean():.2f} | "
      f"Net: ${net_bot:.2f}", "← NET NEGATIVE" if net_bot < 0 else "")
print(f"Top 80%    — Avg LTV: ${top_80['customer_ltv'].mean():.2f} | "
      f"Avg CPA: ${top_acq['acquisition_cost_usd'].mean():.2f} | "
      f"Net: ${net_top:.2f}")

---
# Block 8 — Discounting Deep-Dive Analysis

The brief asks whether discounting works and whether the effect **varies by customer type, channel, region, and landing page**.

In [ ]:
# CELL 8.1 — Discount vs conversion by subgroup

print("=== DISCOUNT EFFECTIVENESS BY SUBGROUP ===")
for grp_col in ['channel', 'observed_region', 'loyalty_tier', 'landing_page']:
    tbl = (abt.groupby([grp_col, 'discount_flag'])['converted_30d']
           .mean().unstack('discount_flag').round(3) * 100)
    tbl.columns = ['No Discount (%)', 'Discount Offered (%)']
    tbl['Uplift (pp)'] = (tbl['Discount Offered (%)'] - tbl['No Discount (%)']).round(1)
    print(f"\n--- {grp_col.upper()} ---")
    print(tbl.to_string())

In [ ]:
# CELL 8.2 — Chi-square: discount × conversion within each channel

print("=== CHI-SQUARE: DISCOUNT FLAG × CONVERSION (per channel) ===")
for ch, grp in abt.groupby('channel'):
    sub = grp[['discount_flag','converted_30d']].dropna()
    ct  = pd.crosstab(sub['discount_flag'], sub['converted_30d'])
    if ct.shape == (2, 2):
        chi2, p, dof, _ = chi2_contingency(ct)
        sig = 'SIGNIFICANT' if p < 0.05 else 'Not significant'
        print(f"  {ch:20s}: chi2={chi2:.3f}, p={p:.4f} → {sig}")

In [ ]:
# CELL 8.3 — Margin sacrifice: avg discount % vs avg revenue lift

disc_rev = (transactions.groupby('discount_pct')['revenue_usd']
            .mean().reset_index().sort_values('discount_pct'))

plt.figure(figsize=(8, 4))
plt.scatter(disc_rev['discount_pct'], disc_rev['revenue_usd'],
            color='steelblue', alpha=0.7, s=40)
plt.xlabel('Discount %'); plt.ylabel('Avg Revenue (USD)')
plt.title('Discount % vs Avg Revenue per Transaction')
plt.tight_layout(); plt.show()

print("\nAvg revenue by discount bucket:")
abt['disc_bucket'] = pd.cut(abt['discount_offered_pct'],
                             bins=[-1,0,5,10,15,25],
                             labels=['0%','1-5%','6-10%','11-15%','16-25%'])
print(abt.groupby('disc_bucket', observed=True)
      .agg(leads=('lead_id','count'),
           conv_rate=('converted_30d','mean'),
           avg_ltv=('customer_ltv','mean'))
      .round(3).to_string())

---
# Block 9 — Campaign Grouping (K-Means on Campaigns)

Clusters the 39 campaigns by spend, CTR, CPL, ROAS, and conversion rate to identify which should be scaled, redesigned, or stopped.

In [ ]:
# CELL 9.1 — Build campaign-level KPI table

camp_leads_n = (leads.groupby('campaign_id')
                .agg(total_leads  = ('lead_id',       'count'),
                     converted    = ('converted_30d', 'sum'))
                .reset_index())
camp_leads_n['conversion_rate'] = camp_leads_n['converted'] / camp_leads_n['total_leads']

camp_revenue = (leads.merge(txn_sum[['customer_id','total_revenue']],
                             on='customer_id', how='left')
                .groupby('campaign_id')['total_revenue'].sum().reset_index()
                .rename(columns={'total_revenue':'camp_revenue'}))

camp_kpis = (campaigns
             .merge(camp_leads_n,  on='campaign_id', how='left')
             .merge(camp_revenue,  on='campaign_id', how='left'))
camp_kpis['CTR']  = camp_kpis['clicks']  / camp_kpis['impressions']
camp_kpis['CPL']  = camp_kpis['spend_usd'] / camp_kpis['total_leads']
camp_kpis['ROAS'] = camp_kpis['camp_revenue'] / camp_kpis['spend_usd']
# Replace inf values (from divide-by-zero) with NaN so they are excluded by dropna()
camp_kpis.replace([np.inf, -np.inf], np.nan, inplace=True)

print("Campaign KPIs:")
print(camp_kpis[['campaign_id','channel','objective','spend_usd',
                  'total_leads','conversion_rate','CTR','CPL','ROAS']].to_string(index=False))

In [ ]:
# CELL 9.2 — Elbow + Silhouette to choose K for campaign clusters
# RobustScaler used because campaign KPIs have significant outliers
# KPI outliers capped at 99th percentile before clustering

camp_feat = camp_kpis[['campaign_id','channel','objective',
                         'spend_usd','CTR','CPL','ROAS','conversion_rate']].dropna()
# Cap extreme values so outliers don't dominate clustering
for col in ['spend_usd','CPL','ROAS']:
    p99c = camp_feat[col].quantile(0.99)
    camp_feat[col] = camp_feat[col].clip(upper=p99c)

from sklearn.preprocessing import RobustScaler
X_camp = RobustScaler().fit_transform(
    camp_feat[['spend_usd','CTR','CPL','ROAS','conversion_rate']])

inertia    = []
sil_scores = []
K_range    = range(2, 7)
for k in K_range:
    km     = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_camp)
    inertia.append(km.inertia_)
    sil_scores.append(silhouette_score(X_camp, labels))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(K_range, inertia, marker='o', color='steelblue')
axes[0].set_title('Campaign Clustering — Elbow Method')
axes[0].set_xlabel('K'); axes[0].set_ylabel('Inertia')

axes[1].plot(K_range, sil_scores, marker='s', color='coral')
axes[1].set_title('Campaign Clustering — Silhouette Scores')
axes[1].set_xlabel('K'); axes[1].set_ylabel('Silhouette Score')
plt.tight_layout(); plt.show()

best_k_camp = K_range[np.argmax(sil_scores)]
print(f"Best K by silhouette: {best_k_camp}  (score={max(sil_scores):.3f})")
print("Silhouette scores:", {k: round(s,3) for k,s in zip(K_range, sil_scores)})

In [ ]:
# CELL 9.3 — Fit campaign clusters and assign business recommendations

km_camp    = KMeans(n_clusters=best_k_camp, random_state=42, n_init=10)
camp_feat  = camp_feat.copy()
camp_feat['cluster'] = km_camp.fit_predict(X_camp)

camp_profile = (camp_feat.groupby('cluster')
                [['spend_usd','CTR','CPL','ROAS','conversion_rate']]
                .mean().round(4))
camp_profile['campaign_count'] = camp_feat['cluster'].value_counts().sort_index()

print("=== CAMPAIGN CLUSTER PROFILES ===")
print(camp_profile.to_string())

# Assign recommendations based on ROAS and conversion rate
def label_camp_cluster(row):
    if row['ROAS'] >= camp_profile['ROAS'].median() and row['conversion_rate'] >= camp_profile['conversion_rate'].median():
        return 'Scale Up'
    elif row['ROAS'] < camp_profile['ROAS'].quantile(0.33):
        return 'Stop / Redesign'
    else:
        return 'Maintain / Optimise'

camp_profile['Recommendation'] = camp_profile.apply(label_camp_cluster, axis=1)
print("\n=== CAMPAIGN CLUSTER RECOMMENDATIONS ===")
print(camp_profile[['ROAS','conversion_rate','campaign_count','Recommendation']].to_string())

# Scatter: ROAS vs CPL coloured by cluster
plt.figure(figsize=(8, 5))
colors_c = ['steelblue','coral','seagreen','mediumpurple']
for cl in sorted(camp_feat['cluster'].unique()):
    sub = camp_feat[camp_feat['cluster'] == cl]
    lbl = camp_profile.loc[cl,'Recommendation']
    plt.scatter(sub['CPL'], sub['ROAS'],
                label=f'Cluster {cl}: {lbl}  (n={len(sub)})',
                color=colors_c[cl % len(colors_c)], s=80, alpha=0.8)
plt.title('Campaign Clusters: ROAS vs CPL')
plt.xlabel('Cost Per Lead (CPL)'); plt.ylabel('ROAS')
plt.legend(); plt.tight_layout(); plt.show()

---
# Block 10 — Customer Segmentation (K-Means)

**Fixes applied:**
- **Behavioral features added** (session quality, bounce rate, add-to-cart) alongside demographic and commercial features
- **Silhouette score** used to validate K selection alongside the elbow method
- **Business labels** assigned to each cluster

In [ ]:
# CELL 10.1 — Prepare features: demographic + commercial + behavioral

income_order  = {'Low':0,'Lower-Middle':1,'Middle':2,'Upper-Middle':3,'High':4}
loyalty_order = {'Bronze':0,'Silver':1,'Gold':2,'Platinum':3}

seg_data = (customers[['customer_id','age','income_band','loyalty_tier']]
            .merge(txn_sum[['customer_id','total_orders','avg_order_value']], on='customer_id', how='left')
            .merge(sess_sum[['customer_id','bounce_rate','session_quality_score','any_cart']],
                   on='customer_id', how='left'))

seg_data['income_enc']      = seg_data['income_band'].map(income_order)
seg_data['loyalty_enc']     = seg_data['loyalty_tier'].map(loyalty_order)
seg_data['total_orders']    = seg_data['total_orders'].fillna(0)
seg_data['avg_order_value'] = seg_data['avg_order_value'].fillna(0)
seg_data['bounce_rate']     = seg_data['bounce_rate'].fillna(seg_data['bounce_rate'].median())
seg_data['session_quality_score'] = seg_data['session_quality_score'].fillna(0)
seg_data['any_cart']        = seg_data['any_cart'].fillna(0)

# Cap AOV outliers
aov_cap = seg_data['avg_order_value'].quantile(0.99)
seg_data['avg_order_value'] = seg_data['avg_order_value'].clip(upper=aov_cap)

feat_cols = ['age','income_enc','loyalty_enc','total_orders',
             'avg_order_value','bounce_rate','session_quality_score','any_cart']
seg_clean = seg_data[['customer_id'] + feat_cols].dropna()
X_scaled  = StandardScaler().fit_transform(seg_clean[feat_cols])

print(f"Customers for clustering: {len(seg_clean)}")
print(f"Features: {feat_cols}")

In [ ]:
# CELL 10.2 — Elbow + Silhouette to choose K

inertia    = []
sil_scores = []
K_range    = range(2, 9)
for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    inertia.append(km.inertia_)
    sil_scores.append(silhouette_score(X_scaled, labels))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(K_range, inertia, marker='o', color='steelblue')
axes[0].set_title('Elbow Method'); axes[0].set_xlabel('K'); axes[0].set_ylabel('Inertia')

axes[1].plot(K_range, sil_scores, marker='s', color='coral')
axes[1].set_title('Silhouette Scores'); axes[1].set_xlabel('K'); axes[1].set_ylabel('Score')
plt.tight_layout(); plt.show()

best_k = K_range[np.argmax(sil_scores)]
print(f"Best K by silhouette score: {best_k}  (score = {max(sil_scores):.3f})")
print("Silhouette scores:", {k: round(s,3) for k,s in zip(K_range, sil_scores)})

In [ ]:
# CELL 10.3 — Fit K-Means and profile clusters

K = best_k   # data-validated choice
km_final = KMeans(n_clusters=K, random_state=42, n_init=10)
seg_clean = seg_clean.copy()
seg_clean['cluster'] = km_final.fit_predict(X_scaled)

print(f"Cluster sizes (K={K}):")
print(seg_clean['cluster'].value_counts().sort_index())

profile = seg_clean.groupby('cluster')[feat_cols].mean().round(3)
profile['size'] = seg_clean['cluster'].value_counts().sort_index()
print("\nCluster Profiles (mean values):")
print(profile.to_string())

In [ ]:
# CELL 10.4 — Assign business labels to each cluster

seg_val = (seg_clean[['customer_id','cluster']]
           .merge(txn_sum[['customer_id','customer_ltv','total_orders','avg_order_value']],
                  on='customer_id', how='left')
           .groupby('cluster')
           .agg(customers  = ('customer_id',    'count'),
                avg_ltv    = ('customer_ltv',   'mean'),
                avg_orders = ('total_orders',   'mean'),
                avg_aov    = ('avg_order_value','mean'))
           .round(2))

# Label each cluster in plain business language
# Logic: high LTV + high orders = loyalists; low LTV + high bounce = at-risk; middle = potential
ltv_med = seg_val['avg_ltv'].median()
ord_med = seg_val['avg_orders'].median()

def assign_label(row):
    if row['avg_ltv'] >= ltv_med and row['avg_orders'] >= ord_med:
        return 'High-Value Loyalists'
    elif row['avg_ltv'] < ltv_med and row['avg_orders'] < ord_med:
        return 'Low-Engagement / At-Risk'
    else:
        return 'Growth Potential'

seg_val['Business Label'] = seg_val.apply(assign_label, axis=1)
print("=== CUSTOMER SEGMENT SUMMARY ===")
print(seg_val.to_string())

In [ ]:
# CELL 10.5 — Visualise: PCA scatter + value bar charts

pca   = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)
label_map = seg_val['Business Label'].to_dict()
colors = ['steelblue','coral','seagreen','mediumpurple','goldenrod']

plt.figure(figsize=(9, 5))
for c in sorted(seg_clean['cluster'].unique()):
    mask = seg_clean['cluster'] == c
    plt.scatter(X_pca[mask,0], X_pca[mask,1],
                label=f"Cluster {c}: {label_map.get(c,'')}  (n={mask.sum()})",
                color=colors[c % len(colors)], alpha=0.5, s=20)
plt.title('Customer Segments — PCA 2D View')
plt.xlabel('PC1'); plt.ylabel('PC2')
plt.legend(fontsize=8); plt.tight_layout(); plt.show()

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, col, title, color in zip(axes,
        ['avg_ltv','avg_orders','avg_aov'],
        ['Avg LTV (USD)','Avg Orders','Avg Order Value (USD)'],
        ['steelblue','coral','seagreen']):
    xlabels = [f"C{i}\n{label_map.get(i,'')[:12]}" for i in seg_val.index]
    ax.bar(xlabels, seg_val[col], color=color, edgecolor='black')
    ax.set_title(title); ax.set_xlabel('Cluster')
plt.suptitle('Customer Segment Value Comparison', y=1.02)
plt.tight_layout(); plt.show()

In [ ]:
# CELL 10.6 — Channel reach per segment
# Which acquisition channels best reach each customer segment?

seg_channel = (seg_clean[['customer_id','cluster']]
               .merge(abt[['customer_id','channel']].dropna(), on='customer_id', how='left')
               .groupby(['cluster','channel'])
               .size().unstack(fill_value=0))
seg_channel_pct = seg_channel.div(seg_channel.sum(axis=1), axis=0).round(3) * 100
print("=== CHANNEL REACH BY SEGMENT (%) ===")
print(seg_channel_pct.to_string())

---
# Block 11 — Lead Conversion Prediction
## Logistic Regression vs Random Forest vs KNN vs Naive Bayes

In [ ]:
# CELL 11.1 — Feature engineering

le2 = LabelEncoder()
ml  = abt.copy()
ml  = ml.merge(campaigns[['campaign_id','channel','objective','creative_type']],
               on='campaign_id', how='left', suffixes=('','_c'))

for col in ['lead_source','observed_region','landing_page',
            'channel','objective','creative_type']:
    if col in ml.columns:
        ml[col + '_enc'] = le2.fit_transform(ml[col].astype(str))

feat_lr = ['lead_score','discount_offered_pct','acquisition_cost_usd',
           'lead_source_enc','observed_region_enc','landing_page_enc',
           'channel_enc','objective_enc','creative_type_enc']

lr_df = ml[feat_lr + ['converted_30d']].dropna()
X_all = lr_df[feat_lr]; y_all = lr_df['converted_30d']
X_train, X_test, y_train, y_test = train_test_split(
    X_all, y_all, test_size=0.25, random_state=42, stratify=y_all)
sc = StandardScaler()
X_train_s = sc.fit_transform(X_train); X_test_s = sc.transform(X_test)
print(f"Train: {len(X_train)} | Test: {len(X_test)}")

In [ ]:
# CELL 11.2 — Train all four models and compare

models = {
    'Logistic Regression': LogisticRegression(max_iter=500, random_state=42),
    'Random Forest'      : RandomForestClassifier(n_estimators=100, random_state=42),
    'KNN (k=5)'          : KNeighborsClassifier(n_neighbors=5),
    'Naive Bayes'        : GaussianNB()
}
results = []; trained_models = {}
for name, model in models.items():
    model.fit(X_train_s, y_train)
    yp = model.predict(X_test_s)
    results.append({'Model': name,
                    'Accuracy' : round(accuracy_score(y_test,  yp), 4),
                    'Precision': round(precision_score(y_test, yp), 4),
                    'Recall'   : round(recall_score(y_test,    yp), 4),
                    'F1 Score' : round(f1_score(y_test,         yp), 4)})
    trained_models[name] = (model, yp)

results_df = pd.DataFrame(results).sort_values('F1 Score', ascending=False)
best_model_name = results_df.iloc[0]['Model']
print("=" * 60)
print("   MODEL COMPARISON — LEAD CONVERSION PREDICTION")
print("=" * 60)
print(results_df.to_string(index=False))
print(f"\nBest model by F1 Score: {best_model_name}")

In [ ]:
# CELL 11.3 — Confusion matrices for all four models

fig, axes = plt.subplots(1, 4, figsize=(20, 4))
for ax, (name, (model, yp)) in zip(axes, trained_models.items()):
    ConfusionMatrixDisplay(confusion_matrix(y_test, yp),
                           display_labels=['Not Conv','Converted']).plot(
        ax=ax, cmap='Blues', colorbar=False)
    ax.set_title(f"{name}\nAcc={accuracy_score(y_test,yp):.3f}", fontsize=9)
plt.suptitle('Confusion Matrices — All Models', y=1.03)
plt.tight_layout(); plt.show()

In [ ]:
# CELL 11.4 — Feature importance: Random Forest + LR coefficients

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
rf_imp = pd.Series(trained_models['Random Forest'][0].feature_importances_,
                   index=feat_lr).sort_values(ascending=True)
rf_imp.plot(kind='barh', ax=axes[0], color='steelblue')
axes[0].set_title('Random Forest — Feature Importances')

lr_coef = pd.Series(trained_models['Logistic Regression'][0].coef_[0],
                    index=feat_lr).sort_values(ascending=True)
colors  = ['seagreen' if c > 0 else 'tomato' for c in lr_coef]
lr_coef.plot(kind='barh', ax=axes[1], color=colors)
axes[1].axvline(0, color='black', linewidth=0.8)
axes[1].set_title('Logistic Regression — Coefficients')
plt.tight_layout(); plt.show()

In [ ]:
# CELL 11.5 — Lead priority scoring using best model

best_model = trained_models[best_model_name][0]
probs      = best_model.predict_proba(X_test_s)[:, 1]
prio_df    = Xv = X_test.copy()
prio_df['conversion_probability'] = probs
prio_df['converted_30d_actual']   = y_test.values
prio_df['priority_tier'] = pd.cut(probs, bins=[0, 0.33, 0.66, 1.0],
                                   labels=['Low','Medium','High'])
summary = (prio_df.groupby('priority_tier', observed=True)
           .agg(leads       = ('converted_30d_actual', 'count'),
                actual_conv = ('converted_30d_actual', 'sum'),
                avg_prob    = ('conversion_probability','mean'))
           .reset_index())
summary['conv_rate'] = (summary['actual_conv'] / summary['leads'] * 100).round(1)
summary['avg_prob']  = (summary['avg_prob'] * 100).round(1)
print(f"Lead Priority Tiers (scored by {best_model_name}):")
print(summary.to_string(index=False))

---
# Block 12 — Repeat Buyer Prediction
## Logistic Regression vs Random Forest

In [ ]:
# CELL 12.1 — Build dataset and train both models

growth = (customers[['customer_id','age','gender','region','loyalty_tier',
                      'income_band','preferred_device']]
          .merge(txn_sum[['customer_id','is_repeat_buyer',
                           'avg_order_value','avg_discount_t']],
                 on='customer_id', how='inner')
          .merge(leads[['customer_id','lead_source','acquisition_cost_usd']]
                 .dropna(subset=['customer_id']).drop_duplicates('customer_id'),
                 on='customer_id', how='left'))

for col in ['gender','region','loyalty_tier','income_band',
            'preferred_device','lead_source']:
    growth[col + '_enc'] = le2.fit_transform(growth[col].astype(str))

feat_g = ['age','avg_order_value','avg_discount_t','acquisition_cost_usd',
          'gender_enc','region_enc','loyalty_tier_enc',
          'income_band_enc','preferred_device_enc','lead_source_enc']

g_df = growth[feat_g + ['is_repeat_buyer']].dropna()
Xg, yg = g_df[feat_g], g_df['is_repeat_buyer']
Xgt, Xgv, ygt, ygv = train_test_split(Xg, yg, test_size=0.25,
                                        random_state=42, stratify=yg)
scg = StandardScaler()
Xgt_s = scg.fit_transform(Xgt); Xgv_s = scg.transform(Xgv)

growth_models = {
    'Logistic Regression': LogisticRegression(max_iter=500, random_state=42),
    'Random Forest'      : RandomForestClassifier(n_estimators=100, random_state=42)
}
growth_results = []; trained_growth = {}
for name, model in growth_models.items():
    model.fit(Xgt_s, ygt)
    yp = model.predict(Xgv_s)
    growth_results.append({'Model'    : name,
                            'Accuracy' : round(accuracy_score(ygv,  yp), 4),
                            'Precision': round(precision_score(ygv, yp), 4),
                            'Recall'   : round(recall_score(ygv,    yp), 4),
                            'F1 Score' : round(f1_score(ygv,         yp), 4)})
    trained_growth[name] = (model, yp)

g_results_df = pd.DataFrame(growth_results).sort_values('F1 Score', ascending=False)
print("=== REPEAT BUYER MODEL COMPARISON ===")
print(g_results_df.to_string(index=False))

In [ ]:
# CELL 12.2 — Feature importance for best repeat buyer model

best_g_name = g_results_df.iloc[0]['Model']
best_g_model = trained_growth[best_g_name][0]

if best_g_name == 'Random Forest':
    imp = pd.Series(best_g_model.feature_importances_, index=feat_g).sort_values(ascending=True)
    colors_g = ['steelblue'] * len(imp)
else:
    imp = pd.Series(best_g_model.coef_[0], index=feat_g).sort_values(ascending=True)
    colors_g = ['seagreen' if c > 0 else 'tomato' for c in imp]

plt.figure(figsize=(9, 5))
imp.plot(kind='barh', color=colors_g)
if best_g_name == 'Logistic Regression':
    plt.axvline(0, color='black', linewidth=0.8)
plt.title(f'{best_g_name} — Repeat Buyer Feature Importance')
plt.tight_layout(); plt.show()
print(imp.sort_values(ascending=False).round(4).to_string())

---
# Block 13 — Executive Reporting Layer

In [ ]:
# CELL 13.1 — Full executive summary

print("=" * 55)
print("       NOVAMART — EXECUTIVE KPI SUMMARY")
print("=" * 55)
print(kpis.to_string(index=False))

print("\n\n=== CHANNEL PERFORMANCE (sorted by conversion rate) ===")
print(kpi_pivot('channel').to_string())

print("\n\n=== CUSTOMER SEGMENT SUMMARY ===")
print(seg_val.to_string())

print("\n\n=== CAMPAIGN CLUSTER RECOMMENDATIONS ===")
print(camp_profile[['ROAS','conversion_rate','count','Recommendation']].to_string())

print("\n\n=== LEAD CONVERSION MODEL COMPARISON ===")
print(results_df.to_string(index=False))

print("\n\n=== REPEAT BUYER MODEL COMPARISON ===")
print(g_results_df.to_string(index=False))

print("\n\n=== PARETO / GINI SUMMARY ===")
print(f"  Gini (LTV)          : {g_overall:.4f}")
print(f"  Top {pct_cust:.1f}% of customers generate 80% of LTV")
print(f"  Bottom 20% net value: ${net_bot:.2f} per customer")
print(f"  Top 80%    net value: ${net_top:.2f} per customer")
print("\nAnalysis complete.")